# PopQA -- Inference, Evaluation & Popularity-Stratified Analysis

This notebook runs the full HalluLens pipeline on PopQA in **closed-book mode**
(no supporting passages provided -- the model relies on parametric memory).

**Dataset:** [PopQA](https://huggingface.co/datasets/akariasai/PopQA) -- 14,267 entity-centric factual QA pairs.
- Single HuggingFace "test" split, partitioned 80/20 into train/test using `split_seed`
- Format: question + list of possible answers + Wikidata metadata (subj, prop, s_pop)
- `s_pop` = subject Wikipedia popularity -- enables popularity-stratified hallucination analysis

**Why PopQA matters for hallucination detection:**
Entity popularity correlates with model knowledge coverage. Obscure entities are more
likely to trigger hallucinations. PopQA lets us quantify this effect and evaluate whether
hallucination detectors work across the popularity spectrum.

**Steps:**
1. Inference -- generate model responses with activation logging
2. Evaluation -- substring match against any valid answer; write `eval_results.json`
3. Inspection -- per-sample view and popularity-stratified analysis

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 -- Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# -- Repo root on path --
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# -- Model --
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# -- Dataset split and sample count --
SPLIT = "test"   # "test" | "train"
N     = None      # None = entire partition
SPLIT_SEED = 42   # Seed for train/test partition

# -- Storage paths --
base_dir          = Path("shared/popqa")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "popqa" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file     = str(task_dir / "eval_results.json")
eval_results_llm_file = str(task_dir / "eval_results_llm.json")

# -- Activation logging --
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# -- Inference settings --
MAX_TOKENS  = 64
TEMPERATURE = 0.0

# -- Batched inference (ModelAdapter path) --
BATCH_SIZE = 8

# -- LLM evaluator --
LLM_EVALUATOR = None   # None -> use class default

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'}, seed={SPLIT_SEED})")
print(f"Batch size        : {BATCH_SIZE or 'disabled (sequential server path)'}")
print(f"Generations file  : {generations_file}")
print(f"Eval (substring)  : {eval_results_file}")
print(f"Eval (LLM judge)  : {eval_results_llm_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 -- Inference

Generates model responses for PopQA questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="popqa",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
    batch_size=BATCH_SIZE,
)

## 3 -- Evaluation

Two evaluation methods:

- **Substring match** (`eval`) -- correct if ANY answer in `possible_answers` appears in the response.
- **LLM judge** (`eval_llm`) -- uses a large language model for semantic correctness assessment.

In [ ]:
from scripts.run_with_server import run_experiment

# -- 3a: Substring match eval --
run_experiment(
    step="eval",
    task="popqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

# -- 3b: LLM-judge eval --
run_experiment(
    step="eval_llm",
    task="popqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_llm_file,
    llm_evaluator=LLM_EVALUATOR,
    resume=True,
)

## 4 -- Results Inspection

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    substr_res = json.load(f)

n = substr_res["total_count"]
substr_correct_count = substr_res["accurate_count"]

print(f"Model          : {model_name}")
print(f"Split          : {SPLIT}  (N={n})")
print()
print(f"{'Metric':<30} {'Substring':>12}")
print("-" * 45)
print(f"{'Correct count':<30} {substr_correct_count:>12}")
print(f"{'Correct rate':<30} {substr_res['correct_rate']:>12.1%}")
print(f"{'Hallucination rate':<30} {substr_res['halu_Rate']:>12.1%}")
print("-" * 45)

## 5 -- Popularity-Stratified Analysis

PopQA includes `s_pop` (subject Wikipedia popularity). This section breaks down
accuracy by popularity bucket to reveal whether the model hallucinates more on
obscure entities.

In [ ]:
import pandas as pd
import json

raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)

# Define popularity buckets
bins = [0, 100, 1000, 10000, 100000, float("inf")]
labels = ["0-100", "100-1K", "1K-10K", "10K-100K", "100K+"]
raw_df["pop_bucket"] = pd.cut(raw_df["s_pop"], bins=bins, labels=labels, right=False)

print("Popularity-Stratified Accuracy")
print("=" * 55)
print(f"{'Popularity':<15} {'Count':>8} {'Correct':>8} {'Hallu':>8} {'Accuracy':>10}")
print("-" * 55)
for label in labels:
    bucket = raw_df[raw_df["pop_bucket"] == label]
    if len(bucket) == 0:
        continue
    correct = bucket["is_correct"].sum()
    total = len(bucket)
    hallu = total - correct
    print(f"{label:<15} {total:>8} {correct:>8} {hallu:>8} {correct/max(1,total):>10.1%}")
print("-" * 55)
print(f"{'TOTAL':<15} {len(raw_df):>8} {raw_df['is_correct'].sum():>8} "
      f"{(~raw_df['is_correct']).sum():>8} {raw_df['is_correct'].mean():>10.1%}")

# Per-relation breakdown (top 10 by count)
print("\nPer-Relation Accuracy (top 10 by count)")
print("=" * 55)
prop_stats = raw_df.groupby("prop").agg(
    count=("is_correct", "size"),
    correct=("is_correct", "sum"),
)
prop_stats["accuracy"] = prop_stats["correct"] / prop_stats["count"]
prop_stats = prop_stats.sort_values("count", ascending=False).head(10)
print(f"{'Relation':<25} {'Count':>8} {'Correct':>8} {'Accuracy':>10}")
print("-" * 55)
for prop, row in prop_stats.iterrows():
    print(f"{str(prop):<25} {int(row['count']):>8} {int(row['correct']):>8} {row['accuracy']:>10.1%}")